In [1]:

# Install DuckDB into the environment used by this notebook.
# Run this once; you don't need to reinstall every session.
%pip install duckdb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 11.2 MB/s  0:00:01 eta 0:00:01

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:

from pathlib import Path
import duckdb
import pandas as pd

# Identify the repository root.
# Assumes Jupyter's working directory is nba_economics/notebooks.
repo_root = Path.cwd().parent

# Locate our frozen Phase A canonical datasets.
canonical_dir = repo_root / "data" / "canonical"

# Establish an in-memory DuckDB database connection.
# We can execute SQL without running a separate database server.
con = duckdb.connect()

print("Canonical directory:", canonical_dir)
print("Directory exists:", canonical_dir.exists())
print("DuckDB version:", duckdb.__version__)


Canonical directory: /Users/jonathantran/Desktop/nba_economics/data/canonical
Directory exists: True
DuckDB version: 2.0.0


In [5]:

# Map SQL view names to canonical CSV filenames.
sources = {
    "seasons": "seasons.csv",                     # C01
    "team_seasons": "team_seasons.csv",           # C04
    "team_apron_seasons": "team_apron_seasons.csv" # C10
}

# Register each canonical CSV as a DuckDB SQL view.
for table_name, filename in sources.items():

    # Construct the complete path to the CSV.
    csv_path = canonical_dir / filename

    # Verify the file exists before creating the view.
    if not csv_path.is_file():
        raise FileNotFoundError(f"Missing canonical file: {csv_path}")

    # Convert the path to a SQL string literal.
    # Escape single quotes so paths containing apostrophes remain valid.
    sql_path = str(csv_path).replace("'", "''")

    # Create a view pointing to the existing CSV.
    # DuckDB will read the file when we query the view.
    con.execute(f"""
        CREATE OR REPLACE VIEW {table_name} AS
        SELECT *
        FROM read_csv_auto('{sql_path}')
    """)

    print(f"Registered SQL view: {table_name}")


Registered SQL view: seasons
Registered SQL view: team_seasons
Registered SQL view: team_apron_seasons


In [6]:

# Inspect each canonical table registered in DuckDB.
for table_name in sources:

    # SQL query to count the rows in each table.
    row_count = con.sql(
        f"SELECT COUNT(*) AS n_rows FROM {table_name}"
    ).fetchone()[0]

    # DESCRIBE returns column names and their DuckDB data types.
    columns = con.sql(
        f"DESCRIBE {table_name}"
    ).df()

    print(f"\n{'=' * 50}")
    print(f"TABLE: {table_name}")
    print(f"ROWS: {row_count}")

    # Display the column names and data types.
    display(columns[["column_name", "column_type"]])


AttributeError: Relations no longer have fetchone(): a relation keeps no result between calls, so every call runs it again. Use rel.limit(1).fetchall() instead.

In [7]:

import duckdb
import sys

# Confirm the versions used by this Jupyter kernel.
print("DuckDB version:", duckdb.__version__)
print("Python version:", sys.version.split()[0])

# Confirm both supported query-result methods work.
print("Single value:", con.execute("SELECT 42").fetchone()[0])
print("DataFrame:", con.sql("SELECT 42 AS answer").df())



DuckDB version: 2.0.0
Python version: 3.14.7
Single value: 42


ResourceInUseException: Resource In Use Error: connection has an unfinished result from execute(); call complete() or abort(), or fetch until a fetch returns no rows, before other work on this connection, or use a separate connection from cursor()

In [8]:

import duckdb
import sys

print("DuckDB version:", duckdb.__version__)
print("Python version:", sys.version.split()[0])

# Fetch all rows so the first query result is fully consumed.
# [0][0] retrieves the first column of the first row.
single_value = con.execute(
    "SELECT 42"
).fetchall()[0][0]

print("Single value:", single_value)

# The connection is now free to execute another query.
result_df = con.sql(
    "SELECT 42 AS answer"
).df()

display(result_df)


DuckDB version: 2.0.0
Python version: 3.14.7
Single value: 42


,answer
0,42


In [9]:

# Inspect each canonical SQL view.
for table_name in sources:

    # Use SQL to count the number of observations.
    # .df() converts the SQL result into a pandas DataFrame.
    row_count = con.sql(
        f"SELECT COUNT(*) AS n_rows FROM {table_name}"
    ).df()["n_rows"].iloc[0]

    # DESCRIBE retrieves the column names and data types.
    columns = con.sql(
        f"DESCRIBE {table_name}"
    ).df()

    print(f"\n{'=' * 50}")
    print(f"TABLE: {table_name}")
    print(f"ROWS: {row_count}")

    # Display each column and its DuckDB data type.
    display(columns[["column_name", "column_type"]])




TABLE: seasons
ROWS: 26


,column_name,column_type
0,season,VARCHAR
1,Year,BIGINT
2,Salary Cap,BIGINT
3,Salary Cap Floor Pct,DOUBLE
4,Salary Cap Floor,BIGINT
5,Luxury Tax,DOUBLE
6,1st Apron,DOUBLE
7,2nd Apron,DOUBLE
8,BAE,BIGINT
9,Non-Taxpayer MLE,BIGINT



TABLE: team_seasons
ROWS: 776


,column_name,column_type
0,team_name,VARCHAR
1,wins,BIGINT
2,losses,BIGINT
3,win_pct,DOUBLE
4,games_behind,DOUBLE
5,points_per_game,DOUBLE
6,opponent_points_per_game,DOUBLE
7,simple_rating_system,DOUBLE
8,conference,VARCHAR
9,season,VARCHAR



TABLE: team_apron_seasons
ROWS: 450


,column_name,column_type
0,franchise_id,BIGINT
1,season,VARCHAR
2,total_apron_allocations,BIGINT
3,first_apron_threshold,BIGINT
4,second_apron_threshold,DOUBLE
5,first_apron_space,BIGINT
6,second_apron_space,DOUBLE
7,above_first_apron,BOOLEAN
8,above_second_apron,BOOLEAN
9,first_apron_hard_capped,BOOLEAN


In [10]:

# Define each table's expected primary key.
keys = {
    "seasons": ["season"],
    "team_seasons": ["franchise_id", "season"],
    "team_apron_seasons": ["franchise_id", "season"]
}

for table_name, key_columns in keys.items():

    # Combine key columns into a SQL GROUP BY expression.
    key_sql = ", ".join(key_columns)

    # Define SQL to find duplicate key combinations.
    sql_duplicates = f"""
    SELECT
        {key_sql},
        COUNT(*) AS n
    FROM {table_name}
    GROUP BY {key_sql}
    HAVING COUNT(*) > 1
    """

    # Execute the SQL and return a pandas DataFrame.
    duplicates = con.sql(sql_duplicates).df()

    # Construct a SQL condition to identify missing key values.
    null_condition = " OR ".join(
        f"{column} IS NULL" for column in key_columns
    )

    # Define SQL to count rows with missing keys.
    sql_missing = f"""
    SELECT
        COUNT(*) AS n
    FROM {table_name}
    WHERE {null_condition}
    """

    # Execute SQL and extract the count from the DataFrame.
    missing_keys = con.sql(sql_missing).df()["n"].iloc[0]

    print(f"\nTABLE: {table_name}")
    print(f"Duplicate key combinations: {len(duplicates)}")
    print(f"Rows with missing keys: {missing_keys}")



TABLE: seasons
Duplicate key combinations: 0
Rows with missing keys: 0

TABLE: team_seasons
Duplicate key combinations: 0
Rows with missing keys: 0

TABLE: team_apron_seasons
Duplicate key combinations: 0
Rows with missing keys: 0


In [11]:

# Define the SQL query separately from its execution.
sql_join_coverage = """
SELECT
    COUNT(*) AS total_team_seasons,

    -- Count C04 rows without a matching C01 season.
    COUNT(*) FILTER (
        WHERE s.season IS NULL
    ) AS missing_season_match,

    -- Count C04 rows without a matching C10 apron record.
    COUNT(*) FILTER (
        WHERE a.franchise_id IS NULL
    ) AS missing_apron_match

FROM team_seasons AS t

LEFT JOIN seasons AS s
    ON t.season = s.season

LEFT JOIN team_apron_seasons AS a
    ON t.franchise_id = a.franchise_id
   AND t.season = a.season
"""

# Execute SQL and return the results as a pandas DataFrame.
join_coverage_df = con.sql(sql_join_coverage).df()

display(join_coverage_df)


,total_team_seasons,missing_season_match,missing_apron_match
0,776,0,326


In [12]:

# Define the SQL that constructs our A01 base table.
sql_a01_base = """
CREATE OR REPLACE TEMP VIEW a01_base AS

SELECT
    -- Preserve every C04 franchise-season column.
    t.*,

    -- Add season-level context from C01.
    s."Year" AS season_year,
    s."Salary Cap" AS salary_cap,
    s."Salary Cap Floor Pct" AS salary_cap_floor_pct,
    s."Salary Cap Floor" AS salary_cap_floor,
    s."Luxury Tax" AS league_luxury_tax_threshold,
    s."1st Apron" AS league_first_apron,
    s."2nd Apron" AS league_second_apron,
    s.cba_id,

    -- Add franchise-specific apron information from C10.
    a.total_apron_allocations,
    a.first_apron_threshold,
    a.second_apron_threshold,
    a.first_apron_space,
    a.second_apron_space,
    a.above_first_apron,
    a.above_second_apron,
    a.first_apron_hard_capped,
    a.second_apron_hard_capped,
    a.second_apron_applicable,
    a.verification_status AS apron_verification_status,
    a.source_coverage AS apron_source_coverage

FROM team_seasons AS t

-- Every franchise-season should match one season record.
LEFT JOIN seasons AS s
    ON t.season = s.season

-- C10 covers only part of our historical period.
LEFT JOIN team_apron_seasons AS a
    ON t.franchise_id = a.franchise_id
   AND t.season = a.season
"""

# Execute the SQL statement to create the DuckDB view.
# This does not create a pandas DataFrame yet.
con.execute(sql_a01_base)

print("Created DuckDB view: a01_base")


Created DuckDB view: a01_base


In [13]:

# Check the resulting table's row count and unique keys.
sql_a01_validation = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (franchise_id, season)) AS unique_franchise_seasons,

    -- Count observations with a matched season-level salary cap.
    COUNT(salary_cap) AS rows_with_salary_cap,

    -- Count observations with C10 apron allocations.
    COUNT(total_apron_allocations) AS rows_with_apron_data

FROM a01_base
"""

# Convert the SQL result into a pandas DataFrame.
a01_validation_df = con.sql(sql_a01_validation).df()

display(a01_validation_df)

# Use Python assertions to enforce our expected structure.
assert a01_validation_df["total_rows"].iloc[0] == 776
assert a01_validation_df["unique_franchise_seasons"].iloc[0] == 776
assert a01_validation_df["rows_with_salary_cap"].iloc[0] == 776
assert a01_validation_df["rows_with_apron_data"].iloc[0] == 450

print("A01 v0 structural validation passed.")


,total_rows,unique_franchise_seasons,rows_with_salary_cap,rows_with_apron_data
0,776,776,776,450


A01 v0 structural validation passed.


In [14]:

# Define the SQL for A01 v1A.
# We use a TEMP VIEW so our canonical CSVs remain unchanged.

sql_a01_v1a = """
CREATE OR REPLACE TEMP VIEW a01_v1a AS

SELECT
    -- Retain all existing columns from A01 base.
    b.*,

    -- 1. Team payroll relative to the league salary cap.
    b.team_payroll * 1.0
        / NULLIF(b.salary_cap, 0)
        AS payroll_to_cap_ratio,

    -- 2. Reported cap hit relative to the salary cap.
    b.salary_cap_hit * 1.0
        / NULLIF(b.salary_cap, 0)
        AS cap_hit_to_cap_ratio,

    -- 3. Taxable payroll relative to the luxury-tax threshold.
    b.taxable_payroll * 1.0
        / NULLIF(b.league_luxury_tax_threshold, 0)
        AS taxable_payroll_to_tax_ratio,

    -- 4. Apron allocations relative to the first-apron threshold.
    b.total_apron_allocations * 1.0
        / NULLIF(b.first_apron_threshold, 0)
        AS first_apron_allocation_ratio,

    -- 5. Second-apron ratio, only when applicable.
    CASE
        WHEN b.second_apron_applicable = TRUE
        THEN b.total_apron_allocations * 1.0
            / NULLIF(b.second_apron_threshold, 0)
        ELSE NULL
    END AS second_apron_allocation_ratio

FROM a01_base AS b
"""

# Execute the CREATE VIEW statement.
con.execute(sql_a01_v1a)

print("A01 v1A view created successfully.")


A01 v1A view created successfully.


In [15]:

# Define a validation query for our five engineered ratios.

sql_v1a_validation = """
SELECT
    -- Structural integrity: A01 must retain its original grain.
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (franchise_id, season)) AS unique_keys,

    -- Feature coverage: COUNT(column) excludes NULL values.
    COUNT(payroll_to_cap_ratio) AS n_payroll_cap,
    COUNT(cap_hit_to_cap_ratio) AS n_cap_hit_cap,
    COUNT(taxable_payroll_to_tax_ratio) AS n_taxable_tax,
    COUNT(first_apron_allocation_ratio) AS n_first_apron,
    COUNT(second_apron_allocation_ratio) AS n_second_apron,

    -- Basic numerical checks.
    ROUND(MIN(payroll_to_cap_ratio), 3) AS min_payroll_cap,
    ROUND(MAX(payroll_to_cap_ratio), 3) AS max_payroll_cap,

    ROUND(MIN(taxable_payroll_to_tax_ratio), 3) AS min_taxable_tax,
    ROUND(MAX(taxable_payroll_to_tax_ratio), 3) AS max_taxable_tax

FROM a01_v1a
"""

# Execute SQL and convert the results into a pandas DataFrame.
v1a_audit = con.sql(sql_v1a_validation).df()

display(v1a_audit)

# Confirm that no franchise-season rows were lost or duplicated.
assert v1a_audit["total_rows"].iloc[0] == 776
assert v1a_audit["unique_keys"].iloc[0] == 776

print("A01 v1A structural validation passed.")


,total_rows,unique_keys,n_payroll_cap,n_cap_hit_cap,n_taxable_tax,n_first_apron,n_second_apron,min_payroll_cap,max_payroll_cap,min_taxable_tax,max_taxable_tax
0,776,776,776,776,450,450,90,0.545,2.622,0.521,1.435


A01 v1A structural validation passed.


In [17]:

# Inspect the ten highest-payroll teams for the 2023-24 season.

sql_v1a_sample = """
SELECT
    season,
    team_name,

    ROUND(payroll_to_cap_ratio, 3) AS payroll_cap,
    ROUND(cap_hit_to_cap_ratio, 3) AS cap_hit_cap,
    ROUND(taxable_payroll_to_tax_ratio, 3) AS taxable_tax,
    ROUND(first_apron_allocation_ratio, 3) AS first_apron,
    ROUND(second_apron_allocation_ratio, 3) AS second_apron

FROM a01_v1a

WHERE season = '2023-2024'

ORDER BY team_payroll DESC

LIMIT 10
"""

v1a_sample = con.sql(sql_v1a_sample).df()

display(v1a_sample)


,season,team_name,payroll_cap,cap_hit_cap,taxable_tax,first_apron,second_apron
0,2023-2024,Golden State Warriors,1.539,1.539,1.244,1.206,1.137
1,2023-2024,Los Angeles Clippers,1.480,1.480,1.210,1.161,1.094
2,2023-2024,Phoenix Suns,1.425,1.425,1.158,1.115,1.051
3,2023-2024,Milwaukee Bucks,1.377,1.377,1.102,1.071,1.010
4,2023-2024,Boston Celtics,1.374,1.374,1.119,1.094,1.032
5,2023-2024,Denver Nuggets,1.330,1.330,1.070,1.033,0.974
6,2023-2024,Miami Heat,1.302,1.302,1.057,1.028,0.969
7,2023-2024,Los Angeles Lakers,1.249,1.249,1.017,0.980,0.924
8,2023-2024,Dallas Mavericks,1.233,1.233,0.996,0.964,0.909
9,2023-2024,New Orleans Pelicans,1.231,1.231,0.998,0.957,0.902


In [18]:

# Inspect the financial amounts, thresholds, and normalized ratios
# for the five highest-payroll teams in 2023-2024.
# This changes only our inspection output, NOT the underlying A01 view.

sql_v1a_interpretation = """
SELECT
    season,
    team_name,

    -- Payroll compared with the league salary cap.
    ROUND(team_payroll / 1000000.0, 2) AS payroll_m,
    ROUND(salary_cap / 1000000.0, 2) AS salary_cap_m,
    ROUND(payroll_to_cap_ratio * 100, 1) AS payroll_cap_pct,

    -- Reported cap hit compared with the league salary cap.
    ROUND(salary_cap_hit / 1000000.0, 2) AS cap_hit_m,
    ROUND(cap_hit_to_cap_ratio * 100, 1) AS cap_hit_cap_pct,

    -- Taxable payroll compared with the luxury-tax threshold.
    ROUND(taxable_payroll / 1000000.0, 2) AS taxable_payroll_m,
    ROUND(league_luxury_tax_threshold / 1000000.0, 2)
        AS tax_threshold_m,
    ROUND(taxable_payroll_to_tax_ratio * 100, 1)
        AS taxable_tax_pct,

    -- Apron allocations compared with first- and second-apron thresholds.
    ROUND(total_apron_allocations / 1000000.0, 2)
        AS apron_allocations_m,
    ROUND(first_apron_threshold / 1000000.0, 2)
        AS first_apron_m,
    ROUND(first_apron_allocation_ratio * 100, 1)
        AS first_apron_pct,

    ROUND(second_apron_threshold / 1000000.0, 2)
        AS second_apron_m,
    ROUND(second_apron_allocation_ratio * 100, 1)
        AS second_apron_pct

FROM a01_v1a

WHERE season = '2023-2024'

ORDER BY team_payroll DESC

LIMIT 5
"""

display(con.sql(sql_v1a_interpretation).df())


,season,team_name,payroll_m,salary_cap_m,payroll_cap_pct,cap_hit_m,cap_hit_cap_pct,taxable_payroll_m,tax_threshold_m,taxable_tax_pct,apron_allocations_m,first_apron_m,first_apron_pct,second_apron_m,second_apron_pct
0,2023-2024,Golden State Warriors,209.35,136.02,153.9,209.35,153.9,205.57,165.29,124.4,207.81,172.35,120.6,182.79,113.7
1,2023-2024,Los Angeles Clippers,201.37,136.02,148.0,201.37,148.0,200.06,165.29,121.0,200.06,172.35,116.1,182.79,109.4
2,2023-2024,Phoenix Suns,193.84,136.02,142.5,193.84,142.5,191.33,165.29,115.8,192.18,172.35,111.5,182.79,105.1
3,2023-2024,Milwaukee Bucks,187.35,136.02,137.7,187.35,137.7,182.23,165.29,110.2,184.57,172.35,107.1,182.79,101.0
4,2023-2024,Boston Celtics,186.94,136.02,137.4,186.94,137.4,184.94,165.29,111.9,188.58,172.35,109.4,182.79,103.2


In [19]:

# Compare reported payroll, actual tax, estimated tax,
# and total estimated cost for a recent season.
# This is a diagnostic SELECT, not a new engineered feature.

sql_cost_definition_check = """
SELECT
    season,
    team_name,

    ROUND(team_payroll / 1000000.0, 2) AS payroll_m,
    ROUND(tax_paid / 1000000.0, 2) AS tax_paid_m,
    ROUND(estimated_tax_bill / 1000000.0, 2) AS estimated_tax_m,
    ROUND(total_estimated_cost / 1000000.0, 2) AS total_estimated_cost_m,

    -- Test whether total estimated cost matches
    -- payroll plus the estimated tax bill.
    ROUND(
        (total_estimated_cost - team_payroll - estimated_tax_bill)
        / 1000000.0, 2
    ) AS estimated_cost_difference_m

FROM a01_v1a

WHERE total_estimated_cost IS NOT NULL

ORDER BY season DESC, team_payroll DESC

LIMIT 15
"""

display(con.sql(sql_cost_definition_check).df())


,season,team_name,payroll_m,tax_paid_m,estimated_tax_m,total_estimated_cost_m,estimated_cost_difference_m
0,2025-2026,Cleveland Cavaliers,211.86,NaN,69.44,281.93,0.63
1,2025-2026,New York Knicks,207.05,NaN,44.44,251.86,0.38
2,2025-2026,Golden State Warriors,205.12,NaN,67.91,273.02,-0.01
3,2025-2026,Houston Rockets,199.78,NaN,7.17,201.95,-5.01
4,2025-2026,Los Angeles Lakers,197.15,NaN,22.27,217.45,-1.96
5,2025-2026,Minnesota Timberwolves,196.04,NaN,6.14,200.08,-2.09
6,2025-2026,Philadelphia 76ers,194.67,NaN,NaN,187.88,NaN
7,2025-2026,Sacramento Kings,193.33,NaN,NaN,187.67,NaN
8,2025-2026,Los Angeles Clippers,191.94,NaN,6.67,196.78,-1.82
9,2025-2026,Toronto Raptors,191.61,NaN,NaN,187.21,NaN


In [20]:

# V1B diagnostic: test plausible arithmetic definitions
# of C04's total_estimated_cost.
#
# This does not change the analytical dataset.
# It only compares existing source columns.

sql_total_cost_audit = """
SELECT
    COUNT(*) AS total_rows,

    -- Coverage of each relevant source field.
    COUNT(total_estimated_cost) AS n_total_cost,
    COUNT(team_payroll) AS n_team_payroll,
    COUNT(taxable_payroll) AS n_taxable_payroll,
    COUNT(tax_paid) AS n_tax_paid,
    COUNT(estimated_tax_bill) AS n_estimated_tax,

    -- Count rows where total cost equals payroll + estimated tax.
    COUNT(*) FILTER (
        WHERE total_estimated_cost IS NOT NULL
          AND estimated_tax_bill IS NOT NULL
          AND ABS(
              total_estimated_cost
              - team_payroll
              - estimated_tax_bill
          ) < 1
    ) AS matches_payroll_plus_est_tax,

    -- Count rows where total cost equals payroll + reported tax.
    COUNT(*) FILTER (
        WHERE total_estimated_cost IS NOT NULL
          AND tax_paid IS NOT NULL
          AND ABS(
              total_estimated_cost
              - team_payroll
              - tax_paid
          ) < 1
    ) AS matches_payroll_plus_tax_paid,

    -- Count rows where total cost equals taxable payroll + estimated tax.
    COUNT(*) FILTER (
        WHERE total_estimated_cost IS NOT NULL
          AND taxable_payroll IS NOT NULL
          AND estimated_tax_bill IS NOT NULL
          AND ABS(
              total_estimated_cost
              - taxable_payroll
              - estimated_tax_bill
          ) < 1
    ) AS matches_taxable_plus_est_tax,

    -- Count rows where total cost equals taxable payroll + reported tax.
    COUNT(*) FILTER (
        WHERE total_estimated_cost IS NOT NULL
          AND taxable_payroll IS NOT NULL
          AND tax_paid IS NOT NULL
          AND ABS(
              total_estimated_cost
              - taxable_payroll
              - tax_paid
          ) < 1
    ) AS matches_taxable_plus_tax_paid

FROM a01_v1a
"""

display(con.sql(sql_total_cost_audit).df())


,total_rows,n_total_cost,n_team_payroll,n_taxable_payroll,n_tax_paid,n_estimated_tax,matches_payroll_plus_est_tax,matches_payroll_plus_tax_paid,matches_taxable_plus_est_tax,matches_taxable_plus_tax_paid
0,776,450,776,450,268,94,0,0,94,0


In [21]:

# Inspect reported total cost alongside the candidate components.
# COALESCE is deliberately avoided here so missing tax
# values are not silently treated as zero.

sql_total_cost_examples = """
SELECT
    season,
    team_name,

    ROUND(team_payroll / 1000000.0, 2) AS payroll_m,
    ROUND(taxable_payroll / 1000000.0, 2) AS taxable_m,
    ROUND(tax_paid / 1000000.0, 2) AS tax_paid_m,
    ROUND(estimated_tax_bill / 1000000.0, 2) AS estimated_tax_m,
    ROUND(total_estimated_cost / 1000000.0, 2) AS total_cost_m,

    ROUND(
        (total_estimated_cost - team_payroll)
        / 1000000.0, 2
    ) AS cost_minus_payroll_m,

    ROUND(
        (total_estimated_cost - taxable_payroll)
        / 1000000.0, 2
    ) AS cost_minus_taxable_m

FROM a01_v1a

WHERE total_estimated_cost IS NOT NULL

ORDER BY season DESC, team_payroll DESC

LIMIT 15
"""

display(con.sql(sql_total_cost_examples).df())


,season,team_name,payroll_m,taxable_m,tax_paid_m,estimated_tax_m,total_cost_m,cost_minus_payroll_m,cost_minus_taxable_m
0,2025-2026,Cleveland Cavaliers,211.86,212.49,NaN,69.44,281.93,70.07,69.44
1,2025-2026,New York Knicks,207.05,207.42,NaN,44.44,251.86,44.81,44.44
2,2025-2026,Golden State Warriors,205.12,205.11,NaN,67.91,273.02,67.90,67.91
3,2025-2026,Houston Rockets,199.78,194.77,NaN,7.17,201.95,2.16,7.17
4,2025-2026,Los Angeles Lakers,197.15,195.18,NaN,22.27,217.45,20.30,22.27
5,2025-2026,Minnesota Timberwolves,196.04,193.94,NaN,6.14,200.08,4.04,6.14
6,2025-2026,Philadelphia 76ers,194.67,187.88,NaN,NaN,187.88,-6.78,0.00
7,2025-2026,Sacramento Kings,193.33,187.67,NaN,NaN,187.67,-5.66,0.00
8,2025-2026,Los Angeles Clippers,191.94,190.12,NaN,6.67,196.78,4.84,6.67
9,2025-2026,Toronto Raptors,191.61,187.21,NaN,NaN,187.21,-4.40,0.00


In [22]:

# Check whether total estimated cost consistently equals:
# taxable payroll + estimated tax (treating missing estimated tax as zero).
#
# COALESCE(x, 0) substitutes zero when x is NULL.
# We use it here to TEST the source's apparent construction rule,
# not to assert that every missing tax estimate represents a true zero liability.

sql_cost_final_check = """
SELECT
    COUNT(*) FILTER (
        WHERE total_estimated_cost IS NOT NULL
    ) AS n_total_cost,

    COUNT(*) FILTER (
        WHERE total_estimated_cost IS NOT NULL
          AND estimated_tax_bill IS NULL
    ) AS n_missing_estimated_tax,

    COUNT(*) FILTER (
        WHERE total_estimated_cost IS NOT NULL
          AND taxable_payroll IS NOT NULL
          AND ABS(
              total_estimated_cost
              - taxable_payroll
              - COALESCE(estimated_tax_bill, 0)
          ) < 1
    ) AS n_formula_matches,

    COUNT(*) FILTER (
        WHERE total_estimated_cost IS NOT NULL
          AND taxable_payroll IS NOT NULL
          AND ABS(
              total_estimated_cost
              - taxable_payroll
              - COALESCE(estimated_tax_bill, 0)
          ) >= 1
    ) AS n_formula_mismatches

FROM a01_v1a
"""

display(con.sql(sql_cost_final_check).df())


,n_total_cost,n_missing_estimated_tax,n_formula_matches,n_formula_mismatches
0,450,356,450,0


In [23]:

# Step 12: Build A01 v1B using the existing v1A view.
# Preserve all v1A columns and add six financial-exposure features.

sql_a01_v1b = """
CREATE OR REPLACE TEMP VIEW a01_v1b AS

SELECT
    b.*,

    -- 1. Signed distance from the luxury-tax threshold.
    -- Positive = above; negative = below.
    b.taxable_payroll
        - b.league_luxury_tax_threshold
        AS tax_threshold_distance,

    -- 2. Signed distance from the first apron.
    b.total_apron_allocations
        - b.first_apron_threshold
        AS first_apron_distance,

    -- 3. Signed distance from the second apron.
    -- Only calculate where the second apron applies.
    CASE
        WHEN b.second_apron_applicable = TRUE
        THEN b.total_apron_allocations
            - b.second_apron_threshold
        ELSE NULL
    END AS second_apron_distance,

    -- 4. Above-tax indicator.
    -- Missing inputs produce NULL, not FALSE.
    CASE
        WHEN b.taxable_payroll IS NULL
          OR b.league_luxury_tax_threshold IS NULL
        THEN NULL
        ELSE b.taxable_payroll > b.league_luxury_tax_threshold
    END AS above_tax_threshold,

    -- 5. Reported tax paid relative to team payroll.
    -- Do not replace missing tax payments with zero.
    b.tax_paid * 1.0
        / NULLIF(b.team_payroll, 0)
        AS tax_paid_to_payroll_ratio,

    -- 6. Estimated total cost relative to taxable payroll.
    -- Both source fields belong to the same accounting framework.
    b.total_estimated_cost * 1.0
        / NULLIF(b.taxable_payroll, 0)
        AS estimated_cost_to_taxable_ratio

FROM a01_v1a AS b
"""

con.execute(sql_a01_v1b)

print("A01 v1B view created successfully.")


A01 v1B view created successfully.


In [24]:

# Validate the franchise-season grain and the six new features.

sql_v1b_validation = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (franchise_id, season)) AS unique_keys,

    -- Coverage for each newly engineered feature.
    COUNT(tax_threshold_distance) AS n_tax_distance,
    COUNT(first_apron_distance) AS n_first_distance,
    COUNT(second_apron_distance) AS n_second_distance,
    COUNT(above_tax_threshold) AS n_above_tax,
    COUNT(tax_paid_to_payroll_ratio) AS n_tax_paid_ratio,
    COUNT(estimated_cost_to_taxable_ratio) AS n_cost_ratio,

    -- Verify the Boolean agrees with the signed distance.
    COUNT(*) FILTER (
        WHERE above_tax_threshold IS NOT NULL
          AND above_tax_threshold
              IS DISTINCT FROM (tax_threshold_distance > 0)
    ) AS tax_flag_mismatches,

    -- Check for unexpected negative tax payments.
    COUNT(*) FILTER (
        WHERE tax_paid_to_payroll_ratio < 0
    ) AS negative_tax_ratios,

    -- Inspect the cost ratio's numerical range.
    ROUND(MIN(estimated_cost_to_taxable_ratio), 3)
        AS min_cost_ratio,
    ROUND(MAX(estimated_cost_to_taxable_ratio), 3)
        AS max_cost_ratio

FROM a01_v1b
"""

v1b_audit = con.sql(sql_v1b_validation).df()
display(v1b_audit)

# Structural checks.
assert v1b_audit["total_rows"].iloc[0] == 776
assert v1b_audit["unique_keys"].iloc[0] == 776

# Consistency checks.
assert v1b_audit["tax_flag_mismatches"].iloc[0] == 0
assert v1b_audit["negative_tax_ratios"].iloc[0] == 0

print("A01 v1B structural and consistency checks passed.")


,total_rows,unique_keys,n_tax_distance,n_first_distance,n_second_distance,n_above_tax,n_tax_paid_ratio,n_cost_ratio,tax_flag_mismatches,negative_tax_ratios,min_cost_ratio,max_cost_ratio
0,776,776,450,450,90,450,268,450,0,0,1.0,1.969


A01 v1B structural and consistency checks passed.


In [25]:

# Inspect financial threshold exposure for the highest-taxable-payroll
# teams in 2023-2024. Dollar values are displayed in millions.
#
# Percentage distances are calculated for presentation only;
# they are not stored as additional A01 features.

sql_v1b_inspection = """
SELECT
    season,
    team_name,

    -- Luxury-tax position.
    ROUND(taxable_payroll / 1e6, 2) AS taxable_m,
    ROUND(league_luxury_tax_threshold / 1e6, 2) AS tax_line_m,
    ROUND(tax_threshold_distance / 1e6, 2) AS tax_distance_m,
    ROUND((taxable_payroll_to_tax_ratio - 1) * 100, 1)
        AS tax_distance_pct,
    above_tax_threshold,

    -- First-apron position.
    ROUND(total_apron_allocations / 1e6, 2) AS allocations_m,
    ROUND(first_apron_threshold / 1e6, 2) AS first_apron_m,
    ROUND(first_apron_distance / 1e6, 2) AS first_distance_m,
    ROUND((first_apron_allocation_ratio - 1) * 100, 1)
        AS first_distance_pct,

    -- Second-apron position.
    ROUND(second_apron_threshold / 1e6, 2) AS second_apron_m,
    ROUND(second_apron_distance / 1e6, 2) AS second_distance_m,
    ROUND((second_apron_allocation_ratio - 1) * 100, 1)
        AS second_distance_pct,

    -- Reported tax and estimated total-cost burdens.
    ROUND(tax_paid / 1e6, 2) AS tax_paid_m,
    ROUND(tax_paid_to_payroll_ratio * 100, 1) AS tax_paid_pct,
    ROUND(total_estimated_cost / 1e6, 2) AS total_cost_m,
    ROUND((estimated_cost_to_taxable_ratio - 1) * 100, 1)
        AS estimated_cost_uplift_pct

FROM a01_v1b

WHERE season = '2023-2024'

ORDER BY taxable_payroll DESC

LIMIT 10
"""

display(con.sql(sql_v1b_inspection).df())


,season,team_name,taxable_m,tax_line_m,tax_distance_m,tax_distance_pct,above_tax_threshold,allocations_m,first_apron_m,first_distance_m,first_distance_pct,second_apron_m,second_distance_m,second_distance_pct,tax_paid_m,tax_paid_pct,total_cost_m,estimated_cost_uplift_pct
0,2023-2024,Golden State Warriors,205.57,165.29,40.28,24.4,True,207.81,172.35,35.47,20.6,182.79,25.02,13.7,NaN,NaN,382.45,86.0
1,2023-2024,Los Angeles Clippers,200.06,165.29,34.77,21.0,True,200.06,172.35,27.71,16.1,182.79,17.27,9.4,NaN,NaN,342.47,71.2
2,2023-2024,Phoenix Suns,191.33,165.29,26.04,15.8,True,192.18,172.35,19.84,11.5,182.79,9.39,5.1,NaN,NaN,259.49,35.6
3,2023-2024,Boston Celtics,184.94,165.29,19.64,11.9,True,188.58,172.35,16.23,9.4,182.79,5.79,3.2,NaN,NaN,228.78,23.7
4,2023-2024,Milwaukee Bucks,182.23,165.29,16.93,10.2,True,184.57,172.35,12.23,7.1,182.79,1.78,1.0,NaN,NaN,234.19,28.5
5,2023-2024,Denver Nuggets,176.89,165.29,11.60,7.0,True,178.09,172.35,5.74,3.3,182.79,-4.70,-2.6,NaN,NaN,197.13,11.4
6,2023-2024,Miami Heat,174.72,165.29,9.43,5.7,True,177.20,172.35,4.85,2.8,182.79,-5.60,-3.1,NaN,NaN,189.98,8.7
7,2023-2024,Los Angeles Lakers,168.06,165.29,2.76,1.7,True,168.96,172.35,-3.39,-2.0,182.79,-13.84,-7.6,NaN,NaN,174.96,4.1
8,2023-2024,New Orleans Pelicans,164.98,165.29,-0.31,-0.2,False,164.90,172.35,-7.44,-4.3,182.79,-17.89,-9.8,NaN,NaN,164.98,0.0
9,2023-2024,Dallas Mavericks,164.63,165.29,-0.66,-0.4,False,166.13,172.35,-6.21,-3.6,182.79,-16.66,-9.1,NaN,NaN,164.63,0.0


A01 v1B — Financial Threshold Exposure

Financial threshold exposure: Taxable payroll and apron allocations are source-reported accounting measurements. Signed threshold distances indicate reported financial positioning, not necessarily official end-of-season assessments or legal hard-cap status.

Tax coverage: tax_paid contains 268 non-null observations, while estimated_tax_bill contains 94. Missing tax values are preserved as missing rather than automatically interpreted as zero.

Estimated total cost: Across 450 observations, total_estimated_cost equals taxable payroll plus the reported estimated-tax component, with missing estimates treated as zero in the source-derived construction. This does not establish that missing estimates represent zero actual liability.

Apron restrictions: Existing C10 hard-cap indicators are preserved with their verification limitations. Apron positioning does not independently establish whether a franchise was hard-capped.

In [26]:

# Inspect the competitive-outcome columns already available in C04.
# This is read-only: it does not change C04 or any A01 view.

sql_c04_competitive_columns = """
SELECT
    column_name,
    data_type
FROM information_schema.columns
WHERE table_name = 'team_seasons'
  AND (
      column_name ILIKE '%win%'
      OR column_name ILIKE '%loss%'
      OR column_name ILIKE '%game%'
      OR column_name ILIKE '%playoff%'
      OR column_name ILIKE '%round%'
      OR column_name ILIKE '%final%'
      OR column_name ILIKE '%champ%'
      OR column_name ILIKE '%seed%'
  )
ORDER BY ordinal_position
"""

display(con.sql(sql_c04_competitive_columns).df())


,column_name,data_type
0,wins,BIGINT
1,losses,BIGINT
2,win_pct,DOUBLE
3,games_behind,DOUBLE
4,points_per_game,DOUBLE
5,opponent_points_per_game,DOUBLE
6,pythagorean_wins,BIGINT
7,pythagorean_losses,BIGINT
8,attendance_per_game,DOUBLE
9,seed,DOUBLE


In [27]:

# Inspect existing playoff-outcome values and coverage.
# This helps us define milestone Booleans without guessing
# how C04 represents playoff advancement.

sql_v1c_playoff_inspection = """
SELECT
    furthest_round,
    round_score,

    COUNT(*) AS n_team_seasons,

    COUNT(playoff_series_won) AS n_series_values,

    MIN(playoff_series_won) AS min_series_won,
    MAX(playoff_series_won) AS max_series_won,

    COUNT(*) FILTER (
        WHERE made_playoffs = TRUE
    ) AS n_playoff_teams,

    COUNT(*) FILTER (
        WHERE made_playoffs = FALSE
    ) AS n_non_playoff_teams,

    COUNT(*) FILTER (
        WHERE made_playoffs IS NULL
    ) AS n_missing_playoff_flags

FROM a01_v1b

GROUP BY
    furthest_round,
    round_score

ORDER BY
    round_score,
    furthest_round
"""

display(con.sql(sql_v1c_playoff_inspection).df())


,furthest_round,round_score,n_team_seasons,n_series_values,min_series_won,max_series_won,n_playoff_teams,n_non_playoff_teams,n_missing_playoff_flags
0,First Round,1.0,208,208,0,0,208,0,0
1,Conference Semifinals,2.0,104,104,1,1,104,0,0
2,Conference Finals,3.0,52,52,2,2,52,0,0
3,NBA Finals,4.0,52,52,3,4,52,0,0
4,NaN,NaN,360,360,0,0,0,360,0


In [28]:

# V1C: confirm regular-season outcome coverage and
# whether Pythagorean wins are usable across the dataset.

sql_v1c_regular_season_audit = """
SELECT
    COUNT(*) AS total_rows,

    COUNT(wins) AS n_wins,
    COUNT(losses) AS n_losses,
    COUNT(win_pct) AS n_win_pct,
    COUNT(pythagorean_wins) AS n_pythagorean_wins,

    -- Check for impossible or inconsistent regular-season values.
    COUNT(*) FILTER (
        WHERE wins < 0 OR losses < 0
    ) AS negative_records,

    COUNT(*) FILTER (
        WHERE wins IS NOT NULL
          AND losses IS NOT NULL
          AND wins + losses = 0
    ) AS zero_game_records,

    -- Compare the canonical winning percentage with
    -- the percentage calculated from wins and losses.
    COUNT(*) FILTER (
        WHERE wins IS NOT NULL
          AND losses IS NOT NULL
          AND win_pct IS NOT NULL
          AND wins + losses > 0
          AND ABS(
              win_pct - wins * 1.0 / (wins + losses)
          ) > 0.001
    ) AS win_pct_mismatches,

    MIN(wins - pythagorean_wins) AS min_wins_above_pyth,
    MAX(wins - pythagorean_wins) AS max_wins_above_pyth

FROM a01_v1b
"""

display(con.sql(sql_v1c_regular_season_audit).df())


,total_rows,n_wins,n_losses,n_win_pct,n_pythagorean_wins,negative_records,zero_game_records,win_pct_mismatches,min_wins_above_pyth,max_wins_above_pyth
0,776,776,776,776,776,0,0,0,-9,8


In [29]:

# A01 v1C — Competitive Outcomes & Spending Efficiency
#
# Build on v1B, preserving all existing canonical columns
# and previously engineered financial features.

sql_a01_v1c = """
CREATE OR REPLACE TEMP VIEW a01_v1c AS
SELECT
    b.*,

    -- 1. Actual regular-season games played.
    -- Useful when comparing seasons with different game counts.
    b.wins + b.losses AS games_played,

    -- 2. Reached at least the conference semifinals (second round).
    CASE
        WHEN b.made_playoffs IS NULL THEN NULL
        WHEN b.made_playoffs = FALSE THEN FALSE
        ELSE b.round_score >= 2
    END AS reached_second_round,

    -- 3. Reached at least the conference finals.
    CASE
        WHEN b.made_playoffs IS NULL THEN NULL
        WHEN b.made_playoffs = FALSE THEN FALSE
        ELSE b.round_score >= 3
    END AS reached_conference_finals,

    -- 4. Reached the NBA Finals.
    CASE
        WHEN b.made_playoffs IS NULL THEN NULL
        WHEN b.made_playoffs = FALSE THEN FALSE
        ELSE b.round_score >= 4
    END AS reached_nba_finals,

    -- 5. Won the NBA championship.
    -- Four playoff series wins identify the champion in our data.
    CASE
        WHEN b.made_playoffs IS NULL THEN NULL
        WHEN b.made_playoffs = FALSE THEN FALSE
        ELSE b.playoff_series_won = 4
    END AS won_championship,

    -- 6. Actual wins minus expected (Pythagorean) wins.
    -- Positive = outperformed scoring-based expectation.
    -- Negative = underperformed scoring-based expectation.
    b.wins - b.pythagorean_wins
        AS wins_above_pythagorean,

    -- 7. Reported team payroll per regular-season victory.
    -- NULLIF prevents division by zero.
    b.team_payroll * 1.0 / NULLIF(b.wins, 0)
        AS payroll_per_win,

    -- 8. Taxable payroll per regular-season victory.
    -- Uses the Spotrac tax-accounting payroll basis.
    b.taxable_payroll * 1.0 / NULLIF(b.wins, 0)
        AS taxable_payroll_per_win,

    -- 9. Total estimated financial cost per regular-season victory.
    -- Includes the source-reported estimated tax component
    -- where present; it is not verified actual ownership spending.
    b.total_estimated_cost * 1.0 / NULLIF(b.wins, 0)
        AS estimated_cost_per_win

FROM a01_v1b AS b
"""

# Register the view in the existing DuckDB connection.
con.execute(sql_a01_v1c)


In [30]:

# V1C structural and semantic audit.
# Checks row preservation, key uniqueness, feature coverage,
# and consistency of the derived playoff milestones.

sql_v1c_validation = """
SELECT
    COUNT(*) AS total_rows,

    COUNT(DISTINCT (franchise_id, season)) AS unique_keys,

    -- Expected full coverage for competitive outcomes.
    COUNT(games_played) AS n_games_played,
    COUNT(reached_second_round) AS n_second_round,
    COUNT(reached_conference_finals) AS n_conf_finals,
    COUNT(reached_nba_finals) AS n_nba_finals,
    COUNT(won_championship) AS n_championship_flags,
    COUNT(wins_above_pythagorean) AS n_pyth_residuals,

    -- Financial coverage depends on underlying source availability.
    COUNT(payroll_per_win) AS n_payroll_per_win,
    COUNT(taxable_payroll_per_win) AS n_taxable_per_win,
    COUNT(estimated_cost_per_win) AS n_estimated_per_win,

    -- These should all be zero.
    COUNT(*) FILTER (
        WHERE reached_conference_finals = TRUE
          AND reached_second_round IS DISTINCT FROM TRUE
    ) AS conf_without_second,

    COUNT(*) FILTER (
        WHERE reached_nba_finals = TRUE
          AND reached_conference_finals IS DISTINCT FROM TRUE
    ) AS finals_without_conf,

    COUNT(*) FILTER (
        WHERE won_championship = TRUE
          AND reached_nba_finals IS DISTINCT FROM TRUE
    ) AS champions_without_finals,

    COUNT(*) FILTER (
        WHERE payroll_per_win < 0
           OR taxable_payroll_per_win < 0
           OR estimated_cost_per_win < 0
    ) AS negative_cost_ratios

FROM a01_v1c
"""

display(con.sql(sql_v1c_validation).df())


,total_rows,unique_keys,n_games_played,n_second_round,n_conf_finals,n_nba_finals,n_championship_flags,n_pyth_residuals,n_payroll_per_win,n_taxable_per_win,n_estimated_per_win,conf_without_second,finals_without_conf,champions_without_finals,negative_cost_ratios
0,776,776,776,776,776,776,776,776,776,450,450,0,0,0,0


In [31]:

# Count cumulative playoff milestones across all 26 seasons.
# Each Boolean is cast to an integer so TRUE counts as 1.

sql_v1c_milestones = """
SELECT
    SUM(CAST(made_playoffs AS INTEGER))
        AS playoff_appearances,

    SUM(CAST(reached_second_round AS INTEGER))
        AS second_round_appearances,

    SUM(CAST(reached_conference_finals AS INTEGER))
        AS conference_finals_appearances,

    SUM(CAST(reached_nba_finals AS INTEGER))
        AS nba_finals_appearances,

    SUM(CAST(won_championship AS INTEGER))
        AS championships

FROM a01_v1c
"""

display(con.sql(sql_v1c_milestones).df())


,playoff_appearances,second_round_appearances,conference_finals_appearances,nba_finals_appearances,championships
0,416.0,208.0,104.0,52.0,26.0


A01 v1 — Methodology Checkpoint

Grain: One franchise-season, keyed by franchise_id and season.
Sources: C04 team seasons, C01 league season thresholds, and C10 team apron seasons.
Feature engineering: 20 derived features across financial normalization (5), financial threshold exposure (6), and competitive outcomes/spending efficiency (9).
Validation: 776 rows, 776 unique keys; agreed structural and consistency checks passed.
Coverage: Taxable payroll and estimated cost measures cover 450 franchise-seasons; reported tax-paid ratios cover 268; second-apron features cover 90 applicable observations.
Accounting limitations: Payroll, taxable payroll, apron allocations, and estimated cost are not interchangeable. Missing estimated tax is not proof of zero liability.
Competitive limitations: Spending per win is descriptive rather than causal. Season lengths differ historically. Pythagorean residuals describe deviations from scoring-based expectations, not necessarily managerial skill.
Persistence: A01 currently exists as a temporary DuckDB view and has not yet been exported.